<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Multi-head Attention Plus Data Loading

In [1]:
# NBVAL_IGNORE_OUTPUT
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.2.2


The complete chapter code is located in [ch03.ipynb](./ch03.ipynb).

This notebook contains the main takeaway, multihead-attention implementation (plus the data loading pipeline from chapter 2)

## Data Loader from Chapter 2

In [2]:
import tiktoken
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        # TODO: tokenize the entire text, then use a sliding window
        # (window size max_length, step stride) to build input_ids and
        # target_ids (each target is its input chunk shifted right by one)
        ...

    def __len__(self):
        # TODO: implement
        ...

    def __getitem__(self, idx):
        # TODO: implement
        ...


def create_dataloader(txt, batch_size=4, max_length=256, stride=128, shuffle=True):
    # Initialize the tokenizer
    # TODO: build a GPTDatasetV1 and wrap it in a DataLoader, then return it
    ...


with open("small-text-sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

vocab_size = 50257
output_dim = 256
max_len = 1024
context_length = max_len


token_embedding_layer = nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

max_length = 4
dataloader = create_dataloader(raw_text, batch_size=8, max_length=max_length, stride=max_length)

In [3]:
for batch in dataloader:
    x, y = batch

    # TODO: compute token + positional embeddings and sum them into input_embeddings
    token_embeddings = ...
    pos_embeddings = ...
    input_embeddings = ...

    break

In [4]:
print(input_embeddings.shape)

torch.Size([8, 4, 256])


# Multi-head Attention from Chapter 3

## Variant A: Simple implementation

In [5]:
class CausalSelfAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        # TODO: create W_query, W_key, W_value (nn.Linear, bias=qkv_bias),
        # a nn.Dropout layer, and register_buffer a causal 'mask'
        # (upper-triangular ones, diagonal=1)
        ...

    def forward(self, x):
        b, n_tokens, d_in = x.shape # New batch dimension b
        # TODO: implement causal self-attention (as in ch03):
        # project to keys/queries/values, attn_scores = queries @ keys.transpose(1, 2),
        # masked_fill_ future positions with -inf, scaled softmax, dropout,
        # return attn_weights @ values
        ...


class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # TODO: create a nn.ModuleList of `num_heads` CausalSelfAttention heads,
        # plus an out_proj = nn.Linear(d_out * num_heads, d_out * num_heads)
        ...

    def forward(self, x):
        # TODO: concatenate all head outputs along the last dim, then apply out_proj
        ...

In [6]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim

num_heads=2
d_out = d_in // num_heads

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads)

batch = input_embeddings
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([8, 4, 256])


## Variant B: Alternative implementation

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        # TODO: create W_query, W_key, W_value (nn.Linear, bias=qkv_bias),
        # out_proj (nn.Linear to combine head outputs), a nn.Dropout layer,
        # and register_buffer the causal 'mask' (upper-triangular ones, diagonal=1)
        ...

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        # TODO: implement multi-head attention with weight splits:
        # 1. project x to keys, queries, values          # Shape: (b, num_tokens, d_out)
        # 2. split last dim into heads via .view          # -> (b, num_tokens, num_heads, head_dim)
        # 3. transpose to                                 # -> (b, num_heads, num_tokens, head_dim)
        # 4. attn_scores = queries @ keys.transpose(2, 3) # dot product per head
        # 5. mask (truncated to num_tokens, as bool) and masked_fill_ with -inf
        # 6. scaled softmax, then dropout
        # 7. context_vec = (attn_weights @ values).transpose(1, 2)
        # 8. combine heads with .contiguous().view(b, num_tokens, self.d_out)
        # 9. apply self.out_proj (optional projection) and return
        ...

In [8]:
torch.manual_seed(123)

context_length = max_length
d_in = output_dim
d_out = d_in

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

batch = input_embeddings
context_vecs = mha(batch)

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([8, 4, 256])
